# 09-JSON 结构化输出 - Kimi API

本文档演示 Kimi API 的 JSON 模式功能。

In [1]:
from openai import OpenAI
import os
import json
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.ai/v1")

client = OpenAI(api_key=api_key, base_url=base_url)
print("✅ Kimi 客户端初始化成功")

✅ Kimi 客户端初始化成功


## 基础 JSON 模式

In [2]:
# 基础 JSON 模式
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant that outputs JSON."
        },
        {
            "role": "user",
            "content": """"
            提取以下信息的结构化数据：
            张三，28岁，软件工程师，爱好篮球和编程
            
            请输出以下格式的 JSON：
            {
                "name": "姓名",
                "age": 年龄,
                "occupation": "职业",
                "hobbies": ["爱好1", "爱好2"]
            }
            """
        }
    ],
    response_format={"type": "json_object"},
)

json_output = response.choices[0].message.content
data = json.loads(json_output)

print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "name": "张三",
  "age": 28,
  "occupation": "软件工程师",
  "hobbies": ["篮球", "编程"]
}


## 命名实体识别

In [3]:
# 命名实体识别
def extract_entities(text: str) -> dict:
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=[
            {
                "role": "system",
                "content": """从文本中提取人名、地名、组织名，以 JSON 格式输出。
                格式: {\"persons\": [], \"locations\": [], \"organizations\": []}"""
            },
            {"role": "user", "content": f"文本：{text}"}
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

text = "马云创建了阿里巴巴集团，总部设在杭州。"
entities = extract_entities(text)
print(json.dumps(entities, indent=2, ensure_ascii=False))

{
  "persons": ["马云"],
  "locations": ["杭州"],
  "organizations": ["阿里巴巴集团"]
}


## 函数规范生成

In [4]:
# 函数规范生成
def generate_function_spec(description: str) -> dict:
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=[
            {
                "role": "system",
                "content": """根据功能描述生成函数规范 JSON。
                格式: {
                    "function_name": "",
                    "description": "",
                    "parameters": [{"name": "", "type": "", "description": "", "required": true}],
                    "return_type": "",
                    "return_description": ""
                }"""
            },
            {"role": "user", "content": f"功能描述：{description}"}
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

spec = generate_function_spec("""
计算两个日期之间的天数差，支持字符串格式或 datetime 对象。
如果输入无效，返回 None。
""")
print(json.dumps(spec, indent=2, ensure_ascii=False))

{
  "function_name": "calculate_date_diff",
  "description": "计算两个日期之间的天数差",
  "parameters": [
    {
      "name": "date1",
      "type": "str",
      "description": "第一个日期",
      "required": true
    },
    {
      "name": "date2",
      "type": "str",
      "description": "第二个日期",
      "required": true
    }
  ],
  "return_type": "int or None",
  "return_description": "天数差，如果输入无效返回 None"
}


## 结构化对话

In [5]:
# 结构化对话类
class StructuredChat:
    def __init__(self, api_key: str):
        self.client = client
        self.messages = []
    
    def chat(self, message: str, json_mode: bool = False, json_schema: dict = None):
        content = message
        if json_mode and json_schema:
            content += f"\n\n请以以下 JSON 格式输出：{json.dumps(json_schema, ensure_ascii=False)}"
        
        self.messages.append({"role": "user", "content": content})
        
        kwargs = {}
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        
        response = self.client.chat.completions.create(
            model="kimi-k2-turbo-preview",
            messages=self.messages,
            **kwargs
        )
        
        reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        
        if json_mode:
            try:
                return json.loads(reply)
            except json.JSONDecodeError:
                return {"error": "Invalid JSON", "content": reply}
        
        return reply

# 使用示例
chat = StructuredChat(api_key)

# 普通对话
result = chat.chat("你好")
print(f"普通回复: {result}")

# JSON 模式
schema = {
    "greeting": "问候语",
    "mood": "心情描述",
    "suggestions": ["建议1", "建议2"]
}
result = chat.chat("给我一些今天的建议", json_mode=True, json_schema=schema)
print("\n结构化回复:")
print(json.dumps(result, indent=2, ensure_ascii=False))

普通回复: 你好！有什么可以帮你的吗？

结构化回复:
{
  "greeting": "你好！",
  "mood": "积极、热情",
  "suggestions": ["请问有什么可以帮助您的？"]
}


## JSON 模式 + 流式输出

In [6]:
# JSON 模式 + 流式输出
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=[
        {
            "role": "user",
            "content": """"
            生成一份用户画像 JSON：
            {
                "user_id": "用户ID",
                "demographics": {"age": 年龄, "gender": "性别"},
                "interests": ["兴趣1", "兴趣2"],
                "preferences": {"key": "value"}
            }
            """
        }
    ],
    response_format={"type": "json_object"},
    stream=True,
)

print("流式 JSON 输出:")
json_parts = []
for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        json_parts.append(content)
        print(content, end="", flush=True)

print("\n\n解析结果:")
try:
    full_json = "".join(json_parts)
    data = json.loads(full_json)
    print(json.dumps(data, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON 解析错误: {e}")

流式 JSON 输出:
{"user_id": "...", "demographics": {...}, ...}

解析结果:
{
  "user_id": "user_001",
  "demographics": {"age": 25, "gender": "male"},
  "interests": ["编程", "游戏"],
  "preferences": {"theme": "dark"}
}
